# Full-pipeline NDCG@10 by model run

This notebook evaluates every TREC-format run in `LLM_re_ranker/outputs/` against the matching `ir_datasets` qrels. It reports graded NDCG@10 for each individual clean or attacked run; it does not alter run files, raw attack outputs, or result summaries.

The required dataset qrels must already be available to `ir_datasets` (for example through `IR_DATASETS_HOME` on the server).

In [1]:
from collections import defaultdict
from functools import lru_cache
from pathlib import Path
import math
import os
import re

import pandas as pd
from IPython.display import display
def find_repository_root(start=Path.cwd()):
    """Return the repository containing the full-pipeline run directory."""
    for candidate in (start, *start.parents):
        if (candidate / 'LLM_re_ranker' / 'outputs').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the repository.')


ROOT = find_repository_root()
os.environ.setdefault('IR_DATASETS_HOME', str(ROOT / 'ir_datasets'))

try:
    import ir_datasets
except ImportError as error:
    raise ImportError(
        'Install ir_datasets in the notebook environment to calculate NDCG@10.'
    ) from error

RUN_DIR = ROOT / 'LLM_re_ranker' / 'outputs'
CUTOFF = 10

DATASET_FROM_FILENAME = {
    'dl19': 'msmarco-passage/trec-dl-2019',
    'trec-dl-2019': 'msmarco-passage/trec-dl-2019',
    'dl20': 'msmarco-passage/trec-dl-2020',
    'trec-dl-2020': 'msmarco-passage/trec-dl-2020',
}

RUN_DIR

WindowsPath('d:/Work/LLM-Ranker-Attack/LLM_re_ranker/outputs')

In [2]:
def dataset_from_filename(path):
    """Infer the ir_datasets identifier from a full-pipeline run filename."""
    name = path.name.lower()
    for token, dataset_name in DATASET_FROM_FILENAME.items():
        if token in name:
            return dataset_name
    return None


def parse_trec_run(path):
    """Read a six-column TREC run into ordered document IDs per query."""
    by_query = defaultdict(list)
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            fields = line.split()
            if len(fields) != 6 or fields[1] != 'Q0':
                continue
            query_id, _, document_id, rank, _, _ = fields
            try:
                by_query[query_id].append((int(rank), document_id))
            except ValueError as error:
                raise ValueError(f'{path}:{line_number}: invalid TREC rank') from error
    return {
        query_id: [document_id for _, document_id in sorted(documents)]
        for query_id, documents in by_query.items()
    }


@lru_cache
def qrels_for(dataset_name):
    """Load graded relevance labels keyed by query and document ID."""
    qrels = defaultdict(dict)
    for qrel in ir_datasets.load(dataset_name).qrels_iter():
        qrels[str(qrel.query_id)][str(qrel.doc_id)] = int(qrel.relevance)
    return qrels


def dcg_at_cutoff(relevances, cutoff=CUTOFF):
    """Calculate graded DCG using the TREC gain and log-discount convention."""
    return sum(
        (2**relevance - 1) / math.log2(rank + 1)
        for rank, relevance in enumerate(relevances[:cutoff], start=1)
    )


def ndcg_at_10(ranking, query_qrels):
    """Calculate one query's graded NDCG@10, or return None if it is unjudged."""
    ideal_relevances = sorted(query_qrels.values(), reverse=True)
    ideal_dcg = dcg_at_cutoff(ideal_relevances)
    if ideal_dcg == 0:
        return None
    observed_relevances = [query_qrels.get(document_id, 0) for document_id in ranking]
    return dcg_at_cutoff(observed_relevances) / ideal_dcg


In [3]:
def run_label(path):
    """Classify the filename only for display; the score uses run contents."""
    name = path.stem.lower()
    if 'clean' in name or re.search(r'(^|[-_.])none($|[-_.])', name):
        return 'Clean'
    if 'defense' in name:
        return 'Defense'
    if re.search(r'(^|[-_.])(so|doh)($|[-_.])', name):
        return 'DOH attack'
    if re.search(r'(^|[-_.])(sd|dch)($|[-_.])', name):
        return 'DCH attack'
    if re.search(r'(^|[-_.])qi($|[-_.])', name):
        return 'Query injection'
    return 'Other run'


def model_from_filename(path):
    """Extract a stable display name from the conventional run filename."""
    name = path.stem
    match = re.match(r'(?P<model>.+?)-(?:dl19|dl20|trec-dl-2019|trec-dl-2020)(?:-|$)', name, re.I)
    return match.group('model') if match else name


records = []
skipped = []
for run_path in sorted(RUN_DIR.glob('*.txt')):
    dataset_name = dataset_from_filename(run_path)
    if dataset_name is None:
        skipped.append((run_path.name, 'dataset not identifiable from filename'))
        continue
    rankings = parse_trec_run(run_path)
    if not rankings:
        skipped.append((run_path.name, 'no six-column TREC records'))
        continue
    qrels = qrels_for(dataset_name)
    per_query = [
        ndcg_at_10(ranking, qrels[query_id])
        for query_id, ranking in rankings.items()
        if query_id in qrels
    ]
    scores = [score for score in per_query if score is not None]
    if not scores:
        skipped.append((run_path.name, 'no judged queries in the run'))
        continue
    records.append({
        'Dataset': dataset_name,
        'Model': model_from_filename(run_path),
        'Run type': run_label(run_path),
        'NDCG@10': sum(scores) / len(scores),
        'Judged queries': len(scores),
        'Run queries': len(rankings),
        'Run file': run_path.relative_to(ROOT).as_posix(),
    })

ndcg_results = pd.DataFrame(records)
if ndcg_results.empty:
    print('No scoreable TREC run files were found.')
else:
    ndcg_results = ndcg_results.sort_values(
        ['Dataset', 'Model', 'Run type', 'Run file']
    ).reset_index(drop=True)
    display(ndcg_results.style.format({'NDCG@10': '{:.4f}'}))

if skipped:
    print('Skipped files:')
    display(pd.DataFrame(skipped, columns=['Run file', 'Reason']))


[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] [starting] https://trec.nist.gov/data/deep/2019qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2019qrels-pass.txt: [00:00] [187kB] [2.54MB/s]


,Dataset,Model,Run type,NDCG@10,Judged queries,Run queries,Run file
0,msmarco-passage/trec-dl-2019,bm25,Other run,0.4364,43,43,LLM_re_ranker/outputs/bm25-dl19-judged43-depth1000.txt
1,msmarco-passage/trec-dl-2019,gpt-oss-120b,Clean,0.6429,43,43,LLM_re_ranker/outputs/gpt-oss-120b-dl19-judged43-clean-20260824-235431.txt
2,msmarco-passage/trec-dl-2019,gpt-oss-120b,Clean,0.5935,1,1,LLM_re_ranker/outputs/gpt-oss-120b-dl19-smoke-20260824-234431-clean.txt
3,msmarco-passage/trec-dl-2019,gpt-oss-120b,DOH attack,0.2393,43,43,LLM_re_ranker/outputs/gpt-oss-120b-dl19-judged43-so-back-20260824-235431.txt
4,msmarco-passage/trec-dl-2019,gpt-oss-120b,DOH attack,0.3229,1,1,LLM_re_ranker/outputs/gpt-oss-120b-dl19-smoke-20260824-234431-so-back.txt
5,msmarco-passage/trec-dl-2019,gpt-oss-20b,Clean,0.6730,38,38,LLM_re_ranker/outputs/gpt-oss-20b-dl19-judged43-clean-20260824-212629.txt
6,msmarco-passage/trec-dl-2019,gpt-oss-20b,DOH attack,0.0928,32,32,LLM_re_ranker/outputs/gpt-oss-20b-dl19-judged43-so-back-20260824-212629.txt
7,msmarco-passage/trec-dl-2019,gpt-oss-20b,Other run,0.4706,14,14,LLM_re_ranker/outputs/gpt-oss-20b-dl19-rescue-input-20260824-212629.txt
8,msmarco-passage/trec-dl-2019,gpt-oss-20b,Other run,0.6339,1,1,LLM_re_ranker/outputs/gpt-oss-20b-dl19-smoke-20260824-210806.txt
9,msmarco-passage/trec-dl-2019,kimi-k2-5,Clean,0.6651,34,34,LLM_re_ranker/outputs/kimi-k2-5-dl19-judged43-clean-20260824-224513.txt


Skipped files:


,Run file,Reason
0,bm25-query-19335.txt,dataset not identifiable from filename
1,gpt-oss-20b-clean-rescue2048-20260824-233045.txt,dataset not identifiable from filename
2,gpt-oss-20b-so-back-rescue2048-20260824-233045...,dataset not identifiable from filename


In [4]:
if not ndcg_results.empty:
    model_summary = (
        ndcg_results.groupby(['Dataset', 'Model', 'Run type'], as_index=False)
        .agg(
            **{'Mean NDCG@10': ('NDCG@10', 'mean')},
            Runs=('Run file', 'count'),
            **{'Mean judged queries': ('Judged queries', 'mean')},
        )
        .sort_values(['Dataset', 'Model', 'Run type'])
    )
    display(
        model_summary.style.format(
            {'Mean NDCG@10': '{:.4f}', 'Mean judged queries': '{:.1f}'}
        )
    )


,Dataset,Model,Run type,Mean NDCG@10,Runs,Mean judged queries
0,msmarco-passage/trec-dl-2019,bm25,Other run,0.4364,1,43.0
1,msmarco-passage/trec-dl-2019,gpt-oss-120b,Clean,0.6182,2,22.0
2,msmarco-passage/trec-dl-2019,gpt-oss-120b,DOH attack,0.2811,2,22.0
3,msmarco-passage/trec-dl-2019,gpt-oss-20b,Clean,0.6730,1,38.0
4,msmarco-passage/trec-dl-2019,gpt-oss-20b,DOH attack,0.0928,1,32.0
5,msmarco-passage/trec-dl-2019,gpt-oss-20b,Other run,0.5523,2,7.5
6,msmarco-passage/trec-dl-2019,kimi-k2-5,Clean,0.6651,1,34.0
7,msmarco-passage/trec-dl-2019,kimi-k2-5,DOH attack,0.7352,1,2.0
8,msmarco-passage/trec-dl-2019,kimi-k2-5,Other run,0.7360,1,1.0
9,msmarco-passage/trec-dl-2019,qwen3-32b,Clean,0.5323,3,11.3
